# T21 — Full Lu-sized capability battery on Qwen3-32B BF16

**Standalone notebook.** Do not rerun the earlier T21 safety generation.

This notebook expands the completed compact capability check to the four capability benchmarks and sample sizes reported by Lu et al.:

| Benchmark | Frozen size |
|---|---:|
| IFEval | 541 |
| MMLU-Pro | 1,400 |
| GSM8K | 1,000 |
| EQ-Bench v2 | 171 |
| **Total / condition** | **3,112** |
| **Two conditions** | **6,224 generations** |

Conditions:
- `UNSTEERED`
- `ASSISTANT_AXIS_CAP_SOURCE_SETTING`

## What is source-matched

- `Qwen/Qwen3-32B`
- exact previously resolved model revision
- BF16, **not quantized**
- `enable_thinking=False`
- pinned Lu source implementation
- released Qwen capping configuration
- `layers_46:54-p0.25`, i.e. layers 46–53

## Important claim boundary

Lu et al. report the four benchmark names and sample sizes, but the exact 1,400 MMLU-Pro and 1,000 GSM8K sampled row membership is not available in the public source artifacts used here. The exact capability prompt/decoding setup is also not fully specified by the Assistant Axis repository.

Therefore this notebook freezes a deterministic public, benchmark-native evaluation and labels it:

`LU_SIZED_PUBLIC_CAPABILITY_REPRODUCTION_NOT_EXACT_SOURCE_MEMBERSHIP`

It must **not** be described as an exact Lu capability replication.

## Reliability safeguards

- Google Drive is mandatory.
- Manifest freezes once and is hash-checked on every rerun.
- Every successful generation is immediately appended to Drive.
- Reruns skip successful `(item_id, condition)` keys.
- Failed keys are retried.
- No 4-bit/8-bit fallback.
- No CPU/disk offload.
- No silent shortening after OOM.
- No `torch.cuda.reset_peak_memory_stats()` calls.
- Qwen3's empty closed `<think></think>` input prefill is correctly allowed under `enable_thinking=False`.

In [ ]:

# 0. Environment.
# Do not install vLLM or a quantization stack: source capping uses HF forward hooks.

import sys, subprocess, importlib.metadata as md

REQ = [
    "transformers==4.57.6",
    "accelerate==1.14.0",
    "huggingface_hub==0.36.2",
    "datasets>=3,<5",
    "safetensors",
    "sentencepiece",
    "psutil",
    "pyyaml",
    "pandas",
    "numpy",
    "scipy",
    "scikit-learn",
    "tqdm",
    "absl-py",
    "langdetect",
    "nltk",
    "immutabledict",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQ])

print("Python:", sys.version.split()[0])
for pkg in ["torch", "transformers", "accelerate", "huggingface_hub", "datasets"]:
    try:
        print(pkg, md.version(pkg))
    except Exception as e:
        print(pkg, "UNKNOWN", repr(e))


In [ ]:

# 1. Exact T21 pins, persistent paths, and helper functions.

from __future__ import annotations

import os, sys, json, time, hashlib, shutil, subprocess, getpass, gc, math, random, re, ast
from pathlib import Path
from contextlib import nullcontext
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd

# ------------------------- EXACT T21 PINS -------------------------
PERSIST_ROOT = Path("/content/drive/MyDrive/[anonymized-repository-name]-t21")
FULL_ROOT = PERSIST_ROOT / "full_capability"
CHECKPOINT_ROOT = PERSIST_ROOT / "checkpoints"
AUDIT_ROOT = PERSIST_ROOT / "audit"
HF_HOME = Path("/content/hf_cache")

MODEL_ID = "Qwen/Qwen3-32B"
MODEL_REVISION = "9216db5781bf21249d130ec9da846c4624c16137"

SOURCE_REPO = "https://github.com/safety-research/assistant-axis.git"
SOURCE_COMMIT = "a98961956072224eaf244eb289d6c01700b63795"
SOURCE_DIR = Path("/content/assistant-axis-source")

CAP_REPO_ID = "lu-christina/assistant-axis-vectors"
CAP_DATASET_REVISION = "3b3b788432ad33e3a28d9ff08e88a530c0740814"
CAP_FILE = "qwen-3-32b/capping_config.pt"
CAP_FILE_SHA256 = "6aec1220487473aaeab80b05d5d960ac54b5dd9080b51ac4bb0bbd1f4330db24"
CAP_EXPERIMENT = "layers_46:54-p0.25"
EXPECTED_CAP_LAYERS = list(range(46, 54))

GOOGLE_RESEARCH_REPO = "https://github.com/google-research/google-research.git"
GOOGLE_RESEARCH_DIR = Path("/content/google-research")

MMLU_REPO = "https://github.com/TIGER-AI-Lab/MMLU-Pro.git"
MMLU_REPO_DIR = Path("/content/MMLU-Pro")

EQBENCH_REPO = "https://github.com/EQ-bench/EQ-Bench.git"
EQBENCH_BRANCH = "main_v2_4"
EQBENCH_DIR = Path("/content/EQ-Bench")

FREEZE_SEED = 210817
N_IFEVAL = 541
N_MMLU = 1400
N_GSM = 1000
N_EQ = 171
EXPECTED_ITEMS = N_IFEVAL + N_MMLU + N_GSM + N_EQ
EXPECTED_GENERATIONS = EXPECTED_ITEMS * 2

MANIFEST_PATH = FULL_ROOT / "full_capability_manifest.jsonl"
FREEZE_META_PATH = FULL_ROOT / "full_capability_freeze.json"
GEN_PATH = CHECKPOINT_ROOT / "t21_full_capability_generations.jsonl"
SUMMARY_PATH = FULL_ROOT / "t21_full_capability_summary.csv"
REPORT_PATH = FULL_ROOT / "t21_full_capability_report.json"
# ----------------------------------------------------------------

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_text(s: str) -> str:
    return sha256_bytes(s.encode("utf-8"))

def sha256_file(path: Path, chunk=8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            x = f.read(chunk)
            if not x:
                break
            h.update(x)
    return h.hexdigest()

def json_dump_atomic(obj, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def write_jsonl_atomic(rows, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def append_jsonl_atomic(row, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())

def read_jsonl(path: Path):
    rows = []
    if not Path(path).exists():
        return rows
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise RuntimeError(f"Corrupt JSONL {path}:{line_no}: {e}")
    return rows

def run(cmd, cwd=None, capture=False):
    return subprocess.run(
        list(cmd),
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture,
    )

def git_sha(path: Path):
    return run(["git", "rev-parse", "HEAD"], cwd=path, capture=True).stdout.strip()

def default_remote_branch(path: Path):
    ref = run(
        ["git", "symbolic-ref", "--short", "refs/remotes/origin/HEAD"],
        cwd=path,
        capture=True,
    ).stdout.strip()
    if not ref.startswith("origin/"):
        raise RuntimeError(f"Unexpected origin/HEAD value: {ref}")
    return ref.split("/", 1)[1]

def ensure_exact_repo(url: str, dest: Path, ref: str | None = None, branch: str | None = None):
    dest = Path(dest)
    if not (dest / ".git").exists():
        if dest.exists():
            shutil.rmtree(dest)
        clone_cmd = ["git", "clone"]
        if branch:
            clone_cmd += ["--branch", branch]
        clone_cmd += [url, str(dest)]
        print("Cloning", url)
        run(clone_cmd)
    else:
        print("Repo already present:", dest)
        run(["git", "fetch", "--all", "--prune"], cwd=dest)

    if ref:
        run(["git", "checkout", "--detach", ref], cwd=dest)
    else:
        use_branch = branch or default_remote_branch(dest)
        run(["git", "checkout", use_branch], cwd=dest)
        run(["git", "pull", "--ff-only"], cwd=dest)
    return git_sha(dest)


def ensure_google_research(ref: str | None = None):
    """
    Sparse/shallow checkout of only instruction_following_eval.
    Avoids cloning the very large full google-research working tree.
    """
    dest = GOOGLE_RESEARCH_DIR
    if not (dest / ".git").exists():
        if dest.exists():
            shutil.rmtree(dest)
        print("Sparse-cloning Google Research instruction_following_eval")
        run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            GOOGLE_RESEARCH_REPO, str(dest)
        ])
        run(["git", "sparse-checkout", "set", "instruction_following_eval"], cwd=dest)
    if ref:
        # Fetch the frozen commit explicitly even if it falls outside a later shallow tip.
        run(["git", "fetch", "--depth", "1", "origin", ref], cwd=dest)
        run(["git", "checkout", "--detach", ref], cwd=dest)
    else:
        branch = default_remote_branch(dest)
        run(["git", "checkout", branch], cwd=dest)
        run(["git", "pull", "--ff-only"], cwd=dest)
    return git_sha(dest)

def load_function_from_source(path: Path, function_name: str, globals_dict=None):
    """
    Load one function definition from a pinned source file via AST without importing
    the source module's unrelated runtime dependencies.
    """
    source = Path(path).read_text(encoding="utf-8")
    tree = ast.parse(source)
    node = next(
        (n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
         and n.name == function_name),
        None,
    )
    if node is None:
        raise RuntimeError(f"Function {function_name} not found in {path}")
    module = ast.Module(body=[node], type_ignores=[])
    ast.fix_missing_locations(module)
    env = dict(globals_dict or {})
    exec(compile(module, str(path), "exec"), env)
    return env[function_name]

def validate_qwen_no_thinking_render(rendered: str):
    """
    Qwen3 may encode enable_thinking=False by pre-filling an EMPTY, already-closed
    think block in the assistant input prefix. That is allowed.
    """
    low = rendered.lower()
    marker = "<|im_start|>assistant"
    pos = low.rfind(marker)
    tail = rendered[pos:] if pos >= 0 else rendered
    tlow = tail.lower()
    opens = tlow.count("<think>")
    closes = tlow.count("</think>")
    if opens == 0 and closes == 0:
        return "NO_THINK_TAGS_IN_GENERATION_PREFIX"
    if opens != closes:
        raise RuntimeError(
            f"Malformed Qwen generation prefix: {opens} <think> vs {closes} </think>."
        )
    blocks = re.findall(r"<think>(.*?)</think>", tail, flags=re.I | re.S)
    if len(blocks) != opens:
        raise RuntimeError("Could not safely parse Qwen think blocks.")
    if any(b.strip() for b in blocks):
        raise RuntimeError("Non-empty reasoning content exists in the no-thinking input prefix.")
    return "EMPTY_THINK_PREFILL_SOURCE_FAITHFUL"

def successful_latest(path: Path):
    latest = {}
    for r in read_jsonl(path):
        key = (r.get("item_id"), r.get("condition"))
        if None in key:
            continue
        if not r.get("technical_error"):
            latest[key] = r
    return latest

# Drive is mandatory for a long run.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    raise RuntimeError(
        "Google Drive mount failed. Stop rather than running a multi-hour job without persistence."
    ) from e

for p in [PERSIST_ROOT, FULL_ROOT, CHECKPOINT_ROOT, AUDIT_ROOT, HF_HOME]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face token (input hidden): ").strip()

source_sha = ensure_exact_repo(SOURCE_REPO, SOURCE_DIR, ref=SOURCE_COMMIT)
if source_sha != SOURCE_COMMIT:
    raise RuntimeError(f"Lu source SHA mismatch: {source_sha} != {SOURCE_COMMIT}")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

print("Persistent root:", PERSIST_ROOT)
print("Pinned Lu source:", source_sha)
print("Expected items:", EXPECTED_ITEMS)
print("Expected two-condition generations:", EXPECTED_GENERATIONS)


In [ ]:

# 2. Freeze the 3,112-item public capability manifest.
# This cell does not inspect capability outcomes.

from datasets import load_dataset
from huggingface_hub import dataset_info
from transformers import AutoTokenizer
import nltk

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

def norm_num(s):
    s = str(s).replace(",", "").strip()
    try:
        x = float(s)
        if abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(x)
    except Exception:
        return s

def preprocess_mmlu_row(row):
    r = dict(row)
    r["options"] = [o for o in r["options"] if o != "N/A"]
    return r

def mmlu_format_cot_example(example, choices, including_answer=True):
    # Mirrors the official evaluate_from_local.py function.
    prompt = "Question:\n"
    prompt += example["question"] + "\n"
    prompt += "Options:\n"
    for i, opt in enumerate(example["options"]):
        prompt += "{}. {}\n".format(choices[i], opt)
    if including_answer:
        cot_content = example["cot_content"].replace(
            "A: Let's think step by step.",
            "Answer: Let's think step by step.",
        )
        prompt += cot_content + "\n\n"
    else:
        prompt += "Answer: Let's think step by step."
    return prompt

def build_mmlu_official_prompt(curr, val_rows, initial_prompt, choices, tokenizer):
    """
    Mirrors official MMLU-Pro 5-shot CoT prompt construction and the official
    4096-context / 2048-generation budgeting rule.
    """
    category = curr["category"]
    category_val = [x for x in val_rows if x["category"] == category]
    k = 5
    while True:
        prompt = initial_prompt.replace("{$}", category) + "\n"
        for ex in category_val[:k]:
            prompt += mmlu_format_cot_example(ex, choices, including_answer=True)
        prompt += mmlu_format_cot_example(curr, choices, including_answer=False)
        token_len = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
        if token_len < 4096 - 2048:
            return prompt, k, token_len
        k -= 1
        if k < 0:
            raise RuntimeError(
                f"MMLU-Pro prompt cannot satisfy official 4096/2048 budget for "
                f"question {curr.get('question_id')}"
            )

if MANIFEST_PATH.exists() != FREEZE_META_PATH.exists():
    raise RuntimeError(
        "Manifest/meta partial state detected. Stop and inspect instead of silently regenerating."
    )

if MANIFEST_PATH.exists():
    freeze_meta = json.loads(FREEZE_META_PATH.read_text(encoding="utf-8"))
    frozen_items = read_jsonl(MANIFEST_PATH)
    observed = sha256_file(MANIFEST_PATH)
    if observed != freeze_meta["manifest_sha256"]:
        raise RuntimeError(
            f"Frozen manifest hash mismatch: {observed} != {freeze_meta['manifest_sha256']}"
        )
    if len(frozen_items) != EXPECTED_ITEMS:
        raise RuntimeError(f"Frozen item count is {len(frozen_items)}, expected {EXPECTED_ITEMS}")

    expected_counts = {
        "IFEval": N_IFEVAL,
        "MMLU-Pro": N_MMLU,
        "GSM8K": N_GSM,
        "EQ-Bench": N_EQ,
    }
    observed_counts = pd.Series([x["benchmark"] for x in frozen_items]).value_counts().to_dict()
    if observed_counts != expected_counts:
        raise RuntimeError(f"Frozen benchmark count mismatch: {observed_counts}")

    # Re-pin evaluator repos for later scoring.
    google_commit = ensure_google_research(ref=freeze_meta["google_research_commit"])
    mmlu_commit = ensure_exact_repo(
        MMLU_REPO, MMLU_REPO_DIR,
        ref=freeze_meta["mmlu_repo_commit"],
    )
    eq_commit = ensure_exact_repo(
        EQBENCH_REPO, EQBENCH_DIR,
        ref=freeze_meta["eqbench_commit"],
    )

    print("Using EXISTING frozen full-capability manifest")
    print("N:", len(frozen_items))
    print("SHA256:", observed)
    print("Counts:", observed_counts)

else:
    print("FIRST FREEZE: resolving public benchmark revisions and exact evaluator commits.")

    google_commit = ensure_google_research()
    mmlu_commit = ensure_exact_repo(MMLU_REPO, MMLU_REPO_DIR)
    eq_commit = ensure_exact_repo(EQBENCH_REPO, EQBENCH_DIR, branch=EQBENCH_BRANCH)

    # ---- IFEval: exact 541 official prompts ----
    ifeval_input = GOOGLE_RESEARCH_DIR / "instruction_following_eval" / "data" / "input_data.jsonl"
    if not ifeval_input.exists():
        raise FileNotFoundError(ifeval_input)
    ifeval_rows = read_jsonl(ifeval_input)
    if len(ifeval_rows) != N_IFEVAL:
        raise RuntimeError(f"Expected {N_IFEVAL} IFEval prompts, found {len(ifeval_rows)}")

    # ---- EQ-Bench v2: exact 171 official questions ----
    eq_questions = EQBENCH_DIR / "data" / "eq_bench_v2_questions_171.json"
    eq_helper = EQBENCH_DIR / "lib" / "run_bench_helper_functions.py"
    eq_scoring = EQBENCH_DIR / "lib" / "scoring.py"
    for p in [eq_questions, eq_helper, eq_scoring]:
        if not p.exists():
            raise FileNotFoundError(p)

    eq_data = json.loads(eq_questions.read_text(encoding="utf-8"))
    if isinstance(eq_data, dict):
        eq_rows = eq_data.get("questions") or eq_data.get("data")
        if eq_rows is None:
            # Some historical variants are dicts keyed by question id.
            eq_rows = list(eq_data.values())
    else:
        eq_rows = eq_data
    if len(eq_rows) != N_EQ:
        raise RuntimeError(f"Expected {N_EQ} EQ-Bench v2 questions, found {len(eq_rows)}")

    eq_remove_revision = load_function_from_source(eq_helper, "remove_revision_instructions")

    # ---- MMLU-Pro/GSM8K: freeze exact HF revisions ----
    # huggingface_hub.dataset_info() does NOT take repo_type here.
    gsm_revision = dataset_info("openai/gsm8k", token=HF_TOKEN).sha
    mmlu_revision = dataset_info("TIGER-Lab/MMLU-Pro", token=HF_TOKEN).sha

    gsm = load_dataset(
        "openai/gsm8k", "main", split="test",
        revision=gsm_revision, token=HF_TOKEN,
    )
    mmlu_test_raw = load_dataset(
        "TIGER-Lab/MMLU-Pro", split="test",
        revision=mmlu_revision, token=HF_TOKEN,
    )
    mmlu_val_raw = load_dataset(
        "TIGER-Lab/MMLU-Pro", split="validation",
        revision=mmlu_revision, token=HF_TOKEN,
    )

    if len(gsm) < N_GSM or len(mmlu_test_raw) < N_MMLU:
        raise RuntimeError(
            f"Public dataset too small: GSM8K={len(gsm)}, MMLU-Pro={len(mmlu_test_raw)}"
        )

    # Tokenizer-only load is cheap and lets us freeze MMLU-Pro's official
    # prompt-length backoff before any capability outcomes are generated.
    freeze_tok = AutoTokenizer.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN, use_fast=True
    )

    initial_prompt_path = MMLU_REPO_DIR / "cot_prompt_lib" / "initial_prompt.txt"
    if not initial_prompt_path.exists():
        raise FileNotFoundError(initial_prompt_path)
    initial_prompt = initial_prompt_path.read_text(encoding="utf-8")

    mmlu_eval_source = MMLU_REPO_DIR / "evaluate_from_local.py"
    if not mmlu_eval_source.exists():
        raise FileNotFoundError(mmlu_eval_source)

    rng = np.random.default_rng(FREEZE_SEED)
    gsm_indices = sorted(rng.choice(len(gsm), size=N_GSM, replace=False).tolist())
    mmlu_indices = sorted(rng.choice(len(mmlu_test_raw), size=N_MMLU, replace=False).tolist())

    mmlu_test = [preprocess_mmlu_row(x) for x in mmlu_test_raw]
    mmlu_val = [preprocess_mmlu_row(x) for x in mmlu_val_raw]
    choices = list("ABCDEFGHIJKLMNOP")

    frozen_items = []

    for i, r in enumerate(ifeval_rows):
        prompt = str(r["prompt"])
        frozen_items.append({
            "item_id": f"ifeval-{i:03d}",
            "benchmark": "IFEval",
            "source_index": i,
            "prompt": prompt,
            "prompt_sha256": sha256_text(prompt),
            "native_metadata": {
                "key": r["key"],
                "instruction_id_list": r["instruction_id_list"],
                "kwargs": r["kwargs"],
            },
        })

    for i, r in enumerate(eq_rows):
        if "prompt" not in r or "reference_answer_fullscale" not in r:
            raise RuntimeError(f"Unexpected EQ-Bench row schema at index {i}")
        prompt = eq_remove_revision(str(r["prompt"]), "en")
        frozen_items.append({
            "item_id": f"eqbench-{i:03d}",
            "benchmark": "EQ-Bench",
            "source_index": i,
            "prompt": prompt,
            "prompt_sha256": sha256_text(prompt),
            "native_metadata": {
                "reference_answer": r.get("reference_answer"),
                "reference_answer_fullscale": r["reference_answer_fullscale"],
            },
        })

    for i in mmlu_indices:
        row = mmlu_test[int(i)]
        prompt, n_shots, raw_prompt_tokens = build_mmlu_official_prompt(
            row, mmlu_val, initial_prompt, choices, freeze_tok
        )
        frozen_items.append({
            "item_id": f"mmlupro-{int(i):05d}",
            "benchmark": "MMLU-Pro",
            "source_index": int(i),
            "prompt": prompt,
            "prompt_sha256": sha256_text(prompt),
            "gold": str(row["answer"]),
            "category": row.get("category"),
            "native_metadata": {
                "question_id": row.get("question_id"),
                "answer_index": row.get("answer_index"),
                "official_cot_shots_used": n_shots,
                "official_raw_prompt_tokens": raw_prompt_tokens,
            },
        })

    for i in gsm_indices:
        row = dict(gsm[int(i)])
        gold = norm_num(str(row["answer"]).split("####")[-1].strip())
        prompt = (
            "Solve the following math word problem. Show enough work to make the answer clear, "
            "and end exactly with 'FINAL: <number>'.\n\n" + str(row["question"])
        )
        frozen_items.append({
            "item_id": f"gsm8k-{int(i):04d}",
            "benchmark": "GSM8K",
            "source_index": int(i),
            "prompt": prompt,
            "prompt_sha256": sha256_text(prompt),
            "gold": gold,
        })

    del freeze_tok

    expected_counts = {
        "IFEval": N_IFEVAL,
        "MMLU-Pro": N_MMLU,
        "GSM8K": N_GSM,
        "EQ-Bench": N_EQ,
    }
    observed_counts = pd.Series([x["benchmark"] for x in frozen_items]).value_counts().to_dict()
    if observed_counts != expected_counts:
        raise RuntimeError(f"Freeze count mismatch: {observed_counts} != {expected_counts}")

    ids = [x["item_id"] for x in frozen_items]
    if len(ids) != len(set(ids)):
        raise RuntimeError("Duplicate item IDs in the frozen capability manifest.")

    write_jsonl_atomic(frozen_items, MANIFEST_PATH)
    manifest_sha = sha256_file(MANIFEST_PATH)

    freeze_meta = {
        "created_at_utc": utcnow(),
        "fidelity_label": "LU_SIZED_PUBLIC_CAPABILITY_REPRODUCTION_NOT_EXACT_SOURCE_MEMBERSHIP",
        "exact_lu_membership_claim_allowed": False,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "source_repo_commit": SOURCE_COMMIT,
        "capping_dataset_revision": CAP_DATASET_REVISION,
        "capping_file_sha256": CAP_FILE_SHA256,
        "capping_experiment": CAP_EXPERIMENT,
        "seed": FREEZE_SEED,
        "manifest_sha256": manifest_sha,
        "n_items": EXPECTED_ITEMS,
        "n_generations_expected": EXPECTED_GENERATIONS,
        "counts": observed_counts,
        "sampling": {
            "IFEval": "ALL_541_OFFICIAL_GOOGLE_RESEARCH_INPUTS",
            "EQ-Bench": "ALL_171_OFFICIAL_EQBENCH_V2_QUESTIONS",
            "MMLU-Pro": "SEEDED_1400_PUBLIC_TEST_ROWS_WITH_OFFICIAL_5SHOT_COT_PROMPT_STYLE",
            "GSM8K": "SEEDED_1000_PUBLIC_TEST_ROWS",
        },
        "google_research_commit": google_commit,
        "ifeval_input_sha256": sha256_file(ifeval_input),
        "mmlu_repo_commit": mmlu_commit,
        "mmlu_evaluate_source_sha256": sha256_file(mmlu_eval_source),
        "mmlu_initial_prompt_sha256": sha256_file(initial_prompt_path),
        "eqbench_commit": eq_commit,
        "eqbench_questions_sha256": sha256_file(eq_questions),
        "eqbench_helper_sha256": sha256_file(eq_helper),
        "eqbench_scoring_sha256": sha256_file(eq_scoring),
        "gsm8k_dataset_revision": gsm_revision,
        "mmlu_pro_dataset_revision": mmlu_revision,
        "claim_boundary": (
            "Sample sizes and benchmark families match Lu et al.; exact Lu MMLU-Pro/GSM8K "
            "membership and exact Assistant Axis capability decoding are not established."
        ),
    }
    json_dump_atomic(freeze_meta, FREEZE_META_PATH)

    if sha256_file(MANIFEST_PATH) != freeze_meta["manifest_sha256"]:
        raise RuntimeError("Manifest bytes changed immediately after freeze.")

    print("FULL CAPABILITY MANIFEST FROZEN")
    print("N:", len(frozen_items))
    print("SHA256:", manifest_sha)
    print("Counts:", observed_counts)

assert len(frozen_items) == EXPECTED_ITEMS


In [ ]:

# 3. Source/evaluator self-tests BEFORE the expensive model load.

# Re-pin all source/evaluator repositories to frozen commits.
assert ensure_exact_repo(SOURCE_REPO, SOURCE_DIR, ref=SOURCE_COMMIT) == SOURCE_COMMIT
assert ensure_google_research(ref=freeze_meta["google_research_commit"]) == freeze_meta["google_research_commit"]
assert ensure_exact_repo(
    MMLU_REPO, MMLU_REPO_DIR,
    ref=freeze_meta["mmlu_repo_commit"],
) == freeze_meta["mmlu_repo_commit"]
assert ensure_exact_repo(
    EQBENCH_REPO, EQBENCH_DIR,
    ref=freeze_meta["eqbench_commit"],
) == freeze_meta["eqbench_commit"]

# Validate official source functions exist without importing EQ-Bench's optional dependencies.
eq_scoring_path = EQBENCH_DIR / "lib" / "scoring.py"
eq_helper_path = EQBENCH_DIR / "lib" / "run_bench_helper_functions.py"

eq_parse_answers = load_function_from_source(
    eq_scoring_path, "parse_answers", globals_dict={"re": re}
)
eq_calculate_score_fullscale = load_function_from_source(
    eq_scoring_path, "calculate_score_fullscale",
    globals_dict={"math": math}
)
eq_remove_revision = load_function_from_source(
    eq_helper_path, "remove_revision_instructions"
)

# Tiny deterministic EQ parser/scorer unit test with official function semantics.
fake_ref = {
    "emotion1": "joy", "emotion1_score": 7,
    "emotion2": "sadness", "emotion2_score": 2,
    "emotion3": "anger", "emotion3_score": 1,
    "emotion4": "fear", "emotion4_score": 3,
}
fake_text = "joy: 7\nsadness: 2\nanger: 1\nfear: 3"
fake_answers, _ = eq_parse_answers(fake_text, False)
fake_score = eq_calculate_score_fullscale(fake_ref, fake_answers)
if fake_score is None or abs(float(fake_score) - 10.0) > 1e-9:
    raise RuntimeError(f"EQ-Bench source-function self-test failed: score={fake_score}")

# Validate MMLU official extraction functions can be loaded from the pinned source.
mmlu_eval_path = MMLU_REPO_DIR / "evaluate_from_local.py"
mmlu_extract_final = load_function_from_source(
    mmlu_eval_path, "extract_final", globals_dict={"re": re}
)
mmlu_extract_again = load_function_from_source(
    mmlu_eval_path, "extract_again",
    globals_dict={"re": re, "extract_final": mmlu_extract_final}
)
mmlu_extract_answer = load_function_from_source(
    mmlu_eval_path, "extract_answer",
    globals_dict={
        "re": re,
        "extract_again": mmlu_extract_again,
        "extract_final": mmlu_extract_final,
    }
)
if mmlu_extract_answer("After reasoning, the answer is (C).") != "C":
    raise RuntimeError("MMLU-Pro official answer extractor self-test failed.")

# Validate IFEval source data still hashes exactly to the freeze.
ifeval_input = GOOGLE_RESEARCH_DIR / "instruction_following_eval" / "data" / "input_data.jsonl"
if sha256_file(ifeval_input) != freeze_meta["ifeval_input_sha256"]:
    raise RuntimeError("IFEval source file hash differs from frozen provenance.")

# Manifest integrity and no accidental overlap/duplication.
if sha256_file(MANIFEST_PATH) != freeze_meta["manifest_sha256"]:
    raise RuntimeError("Frozen capability manifest hash mismatch.")
ids = [x["item_id"] for x in frozen_items]
if len(ids) != len(set(ids)):
    raise RuntimeError("Duplicate item IDs detected.")

print("SOURCE/EVALUATOR SELF-TESTS: PASS")
print("EQ-Bench official parser/scorer: PASS")
print("MMLU-Pro official answer extractor: PASS")
print("IFEval frozen source hash: PASS")
print("Frozen manifest integrity: PASS")


In [ ]:

# 4. High-memory GPU preflight + exact Qwen3-32B BF16/source-capping load.
# If all 6,224 generations already exist, this cell deliberately skips the model load.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download

successful = successful_latest(GEN_PATH)
print("Persistent successful keys:", len(successful), "/", EXPECTED_GENERATIONS)

SKIP_MODEL_LOAD = len(successful) == EXPECTED_GENERATIONS

if SKIP_MODEL_LOAD:
    print("All expected generations already exist. Skipping Qwen load.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU visible. Select G4/H100/A100-80GB before continuing.")

    smi = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.total,memory.used,memory.free",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    ).strip()
    print(smi)

    gpus = []
    for line in smi.splitlines():
        p = [x.strip() for x in line.split(",")]
        gpus.append({
            "index": int(p[0]), "name": p[1],
            "total_MiB": float(p[2]), "used_MiB": float(p[3]), "free_MiB": float(p[4]),
        })
    max_total_gib = max(x["total_MiB"] for x in gpus) / 1024
    if max_total_gib < 76:
        raise RuntimeError(
            f"Largest GPU has {max_total_gib:.1f} GiB VRAM. "
            "Do not load the BF16 32B source model on this allocation."
        )

    free_disk_gib = shutil.disk_usage("/content").free / (1024**3)
    print(f"/content free disk: {free_disk_gib:.1f} GiB")
    if free_disk_gib < 72:
        raise RuntimeError(
            "Less than 72 GiB local disk free. Stop before downloading Qwen3-32B."
        )

    cap_path = Path(hf_hub_download(
        repo_id=CAP_REPO_ID,
        filename=CAP_FILE,
        repo_type="dataset",
        revision=CAP_DATASET_REVISION,
        token=HF_TOKEN,
    ))
    cap_sha = sha256_file(cap_path)
    if cap_sha != CAP_FILE_SHA256:
        raise RuntimeError(f"Capping config SHA mismatch: {cap_sha}")

    from assistant_axis import load_capping_config, build_capping_steerer
    capping_config = load_capping_config(str(cap_path))
    exp = next(
        (e for e in capping_config["experiments"] if e.get("id") == CAP_EXPERIMENT),
        None,
    )
    if exp is None:
        raise RuntimeError(f"Missing capping experiment: {CAP_EXPERIMENT}")
    observed_layers = sorted(
        int(capping_config["vectors"][x["vector"]]["layer"])
        for x in exp["interventions"] if "cap" in x
    )
    if observed_layers != EXPECTED_CAP_LAYERS:
        raise RuntimeError(f"Capping layers mismatch: {observed_layers}")

    tok = AutoTokenizer.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN, use_fast=True
    )
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    print("Loading exact Qwen3-32B revision in BF16...")
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=HF_TOKEN,
        dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
        use_safetensors=True,
        trust_remote_code=False,
    ).eval()
    print(f"Load time: {(time.time() - t0)/60:.2f} min")

    hf_map = getattr(model, "hf_device_map", {}) or {}
    map_devices = {str(v) for v in hf_map.values()}
    param_devices = {str(p.device) for p in model.parameters()}
    offloaded = any(
        ("cpu" in d.lower()) or ("disk" in d.lower()) or ("meta" in d.lower())
        for d in (map_devices | param_devices)
    )
    print("hf_device_map:", sorted(map_devices))
    print("parameter devices:", sorted(param_devices))
    if offloaded:
        raise RuntimeError(
            "BF16 model was offloaded to CPU/disk/meta. This run is not source-comparable."
        )

    # Cheap no-thinking inference probe.
    probe_messages = [{"role": "user", "content": "Give one short reason replication matters."}]
    rendered = tok.apply_chat_template(
        probe_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    render_verdict = validate_qwen_no_thinking_render(rendered)
    enc = tok.apply_chat_template(
        probe_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True,
    )
    dev = model.get_input_embeddings().weight.device
    enc = {k: v.to(dev) for k, v in enc.items()}
    n0 = int(enc["input_ids"].shape[1])
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    probe = tok.decode(out[0, n0:], skip_special_tokens=True)
    if not probe.strip():
        raise RuntimeError("Qwen BF16 probe was empty.")
    if "<think>" in probe.lower() or "</think>" in probe.lower():
        raise RuntimeError("Qwen generated think tags despite enable_thinking=False.")

    json_dump_atomic(
        {
            "created_at_utc": utcnow(),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "precision": "bfloat16",
            "quantized": False,
            "thinking_enabled": False,
            "no_thinking_render_verdict": render_verdict,
            "source_repo_commit": SOURCE_COMMIT,
            "capping_dataset_revision": CAP_DATASET_REVISION,
            "capping_config_sha256": cap_sha,
            "capping_experiment": CAP_EXPERIMENT,
            "capping_layers": observed_layers,
            "offloaded": offloaded,
            "hf_device_map_devices": sorted(map_devices),
            "parameter_devices": sorted(param_devices),
            "probe_preview": probe[:300],
        },
        AUDIT_ROOT / "t21_full_capability_model_load.json",
    )
    print("MODEL/CAPPING LOAD: PASS")


In [ ]:

# 5. Full generation with row-level Drive checkpointing and resume.

if SKIP_MODEL_LOAD:
    print("Generation is already complete. Continue to the release/scoring cells.")
else:
    # Reuse exact official source functions loaded in Cell 3.
    # Recreate them here defensively in case the cell was run independently.
    eq_scoring_path = EQBENCH_DIR / "lib" / "scoring.py"
    eq_parse_answers = load_function_from_source(
        eq_scoring_path, "parse_answers", globals_dict={"re": re}
    )
    eq_calculate_score_fullscale = load_function_from_source(
        eq_scoring_path, "calculate_score_fullscale", globals_dict={"math": math}
    )

    mmlu_eval_path = MMLU_REPO_DIR / "evaluate_from_local.py"
    mmlu_extract_final = load_function_from_source(
        mmlu_eval_path, "extract_final", globals_dict={"re": re}
    )
    mmlu_extract_again = load_function_from_source(
        mmlu_eval_path, "extract_again",
        globals_dict={"re": re, "extract_final": mmlu_extract_final}
    )
    mmlu_extract_answer = load_function_from_source(
        mmlu_eval_path, "extract_answer",
        globals_dict={
            "re": re,
            "extract_again": mmlu_extract_again,
            "extract_final": mmlu_extract_final,
        }
    )

    def norm_num(s):
        s = str(s).replace(",", "").strip()
        try:
            x = float(s)
            if abs(x - round(x)) < 1e-9:
                return str(int(round(x)))
            return str(x)
        except Exception:
            return s

    def parse_gsm(text):
        m = re.findall(
            r"FINAL\s*:\s*([-+]?\d[\d,]*(?:\.\d+)?)",
            str(text), flags=re.I
        )
        if m:
            return norm_num(m[-1])
        nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", str(text))
        return norm_num(nums[-1]) if nums else None

    def render_encode(prompt):
        messages = [{"role": "user", "content": str(prompt)}]
        rendered = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        validate_qwen_no_thinking_render(rendered)
        enc = tok.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_tensors="pt",
            return_dict=True,
        )
        dev = model.get_input_embeddings().weight.device
        return rendered, {k: v.to(dev) for k, v in enc.items()}

    def params_for(benchmark, attempt):
        # Benchmark-native/public decoding where available.
        if benchmark == "IFEval":
            return {
                "max_new_tokens": 1024,
                "do_sample": False,
                "temperature": None,
                "top_p": None,
            }
        if benchmark == "MMLU-Pro":
            # Official MMLU-Pro local evaluator: temperature=0, max_tokens=2048.
            return {
                "max_new_tokens": 2048,
                "do_sample": False,
                "temperature": None,
                "top_p": None,
            }
        if benchmark == "GSM8K":
            return {
                "max_new_tokens": 512,
                "do_sample": False,
                "temperature": None,
                "top_p": None,
            }
        if benchmark == "EQ-Bench":
            # Official v2 process_question starts at temp=.01 and +.15 on parse retry.
            return {
                "max_new_tokens": 60,
                "do_sample": True,
                "temperature": 0.01 + 0.15 * attempt,
                "top_p": 1.0,
            }
        raise ValueError(benchmark)

    items_by_bench = defaultdict(list)
    for item in frozen_items:
        items_by_bench[item["benchmark"]].append(item)

    # Finish whole small/full benchmark sets first to maximize useful output if runtime ends.
    BENCHMARK_ORDER = ["EQ-Bench", "IFEval", "MMLU-Pro", "GSM8K"]
    CONDITIONS = ["UNSTEERED", "ASSISTANT_AXIS_CAP_SOURCE_SETTING"]

    successful = successful_latest(GEN_PATH)
    print("Resume status:", len(successful), "/", EXPECTED_GENERATIONS)

    session_start = time.time()
    session_success = 0
    session_times = []

    def show_progress(force=False):
        if not force and session_success % 25 != 0:
            return
        elapsed = max(time.time() - session_start, 1e-9)
        rate = session_success / elapsed
        remaining = EXPECTED_GENERATIONS - len(successful)
        eta_h = (remaining / rate / 3600) if rate > 0 else float("inf")
        median_s = float(np.median(session_times[-100:])) if session_times else None
        print(
            f"[progress] {len(successful)}/{EXPECTED_GENERATIONS} successful | "
            f"session={session_success} | "
            f"median_last100={median_s:.1f}s/gen | ETA≈{eta_h:.2f}h"
            if median_s is not None else
            f"[progress] {len(successful)}/{EXPECTED_GENERATIONS} successful"
        )

    for benchmark in BENCHMARK_ORDER:
        bench_items = items_by_bench[benchmark]
        print(f"\n===== {benchmark}: {len(bench_items)} frozen items =====")

        for condition in CONDITIONS:
            capped = condition == "ASSISTANT_AXIS_CAP_SOURCE_SETTING"
            print(f"\n--- {condition} ---")

            # Register Lu's source hooks once for the full condition block.
            ctx = build_capping_steerer(
                model, capping_config, CAP_EXPERIMENT
            ) if capped else nullcontext()

            with ctx:
                for item in bench_items:
                    key = (item["item_id"], condition)
                    if key in successful:
                        continue

                    seed = FREEZE_SEED * 100000 + int(item["source_index"])
                    attempts = 5 if benchmark == "EQ-Bench" else 2
                    success_row = None
                    last_error = None

                    for attempt in range(attempts):
                        p = params_for(benchmark, attempt)
                        try:
                            rendered, enc = render_encode(item["prompt"])
                            n0 = int(enc["input_ids"].shape[1])

                            torch.manual_seed(seed + attempt)
                            torch.cuda.manual_seed_all(seed + attempt)

                            kwargs = dict(
                                **enc,
                                max_new_tokens=p["max_new_tokens"],
                                do_sample=p["do_sample"],
                                pad_token_id=tok.pad_token_id,
                                eos_token_id=tok.eos_token_id,
                            )
                            if p["do_sample"]:
                                kwargs["temperature"] = p["temperature"]
                                kwargs["top_p"] = p["top_p"]

                            t0 = time.time()
                            with torch.inference_mode():
                                out = model.generate(**kwargs)
                            gen_s = time.time() - t0
                            n_new = int(out.shape[1] - n0)
                            text = tok.decode(out[0, n0:], skip_special_tokens=True)

                            if not text.strip():
                                raise RuntimeError("EMPTY_COMPLETION")
                            if "<think>" in text.lower() or "</think>" in text.lower():
                                raise RuntimeError(
                                    "UNEXPECTED_THINK_TAGS_IN_GENERATED_COMPLETION"
                                )

                            pred = None
                            correct = None
                            parseable = True
                            eq_score = None
                            parse_error = None

                            if benchmark == "MMLU-Pro":
                                pred = mmlu_extract_answer(text)
                                parseable = pred is not None
                                correct = bool(pred == str(item["gold"]))
                            elif benchmark == "GSM8K":
                                pred = parse_gsm(text)
                                parseable = pred is not None
                                correct = bool(pred == norm_num(item["gold"]))
                            elif benchmark == "EQ-Bench":
                                answers, _ = eq_parse_answers(text, False)
                                ref = item["native_metadata"]["reference_answer_fullscale"]
                                eq_score = eq_calculate_score_fullscale(ref, answers)
                                parseable = eq_score is not None
                                if not parseable:
                                    parse_error = "OFFICIAL_EQBENCH_PARSE_FAILED"
                                    if attempt < attempts - 1:
                                        last_error = parse_error
                                        continue

                            success_row = {
                                "created_at_utc": utcnow(),
                                "item_id": item["item_id"],
                                "benchmark": benchmark,
                                "condition": condition,
                                "source_index": item["source_index"],
                                "model_id": MODEL_ID,
                                "model_revision": MODEL_REVISION,
                                "precision": "bfloat16",
                                "quantized": False,
                                "thinking_enabled": False,
                                "source_repo_commit": SOURCE_COMMIT,
                                "capping_dataset_revision": CAP_DATASET_REVISION,
                                "capping_config_sha256": CAP_FILE_SHA256 if capped else None,
                                "capping_experiment": CAP_EXPERIMENT if capped else None,
                                "manifest_sha256": freeze_meta["manifest_sha256"],
                                "prompt_sha256": item["prompt_sha256"],
                                "seed": seed,
                                "attempt": attempt,
                                "do_sample": p["do_sample"],
                                "temperature": p["temperature"],
                                "top_p": p["top_p"],
                                "max_new_tokens": p["max_new_tokens"],
                                "prompt_tokens": n0,
                                "generated_tokens": n_new,
                                "generation_seconds": gen_s,
                                "tokens_per_second": n_new / max(gen_s, 1e-9),
                                "completion": text,
                                "prediction": pred,
                                "gold": item.get("gold"),
                                "correct": correct,
                                "parseable": bool(parseable),
                                "eqbench_item_score": eq_score,
                                "parse_error": parse_error,
                                "response_chars": len(text),
                                "technical_error": None,
                            }
                            break

                        except torch.cuda.OutOfMemoryError as e:
                            last_error = f"CUDA_OOM: {e}"
                            gc.collect()
                            torch.cuda.empty_cache()
                            time.sleep(2)
                        except Exception as e:
                            last_error = repr(e)
                            time.sleep(min(8, 2 ** attempt))

                    if success_row is None:
                        append_jsonl_atomic(
                            {
                                "created_at_utc": utcnow(),
                                "item_id": item["item_id"],
                                "benchmark": benchmark,
                                "condition": condition,
                                "source_index": item["source_index"],
                                "manifest_sha256": freeze_meta["manifest_sha256"],
                                "technical_error": last_error or "UNKNOWN_FAILURE",
                            },
                            GEN_PATH,
                        )
                    else:
                        append_jsonl_atomic(success_row, GEN_PATH)
                        successful[key] = success_row
                        session_success += 1
                        session_times.append(float(success_row["generation_seconds"]))
                        show_progress()

            # Durable block checkpoint summary after each benchmark × condition.
            json_dump_atomic(
                {
                    "created_at_utc": utcnow(),
                    "benchmark": benchmark,
                    "condition": condition,
                    "persistent_successful_keys": len(successful),
                    "expected_total": EXPECTED_GENERATIONS,
                },
                FULL_ROOT / f"progress_{benchmark.replace('-', '').lower()}_{condition.lower()}.json",
            )

    show_progress(force=True)

    missing = []
    for item in frozen_items:
        for condition in CONDITIONS:
            if (item["item_id"], condition) not in successful:
                missing.append([item["item_id"], condition])

    print("\nSuccessful keys:", len(successful), "/", EXPECTED_GENERATIONS)
    if missing:
        json_dump_atomic(
            {
                "created_at_utc": utcnow(),
                "missing_count": len(missing),
                "missing_preview": missing[:100],
                "instruction": (
                    "Rerun this notebook/cell. Successful rows are skipped; only missing keys retry."
                ),
            },
            FULL_ROOT / "full_capability_missing_after_run.json",
        )
        raise RuntimeError(
            f"{len(missing)} keys remain missing. Completed work is safe on Drive. "
            "Rerun to resume; do not alter the frozen manifest."
        )

    print("FULL GENERATION COMPLETE: 6,224 / 6,224 successful keys")


In [ ]:

# 6. Release GPU memory after generation.
# Safe to run once the generation cell has finished or after all 6,224 keys already existed.

for name in ["model", "tok", "capping_config"]:
    if name in globals():
        del globals()[name]

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

print("GPU model objects released. Persistent capability generations remain at:", GEN_PATH)


In [ ]:

# 7. Score all four benchmarks using native/public metrics.
# CPU-only once the 6,224 generation keys exist.

latest = successful_latest(GEN_PATH)
print("Scorable successful keys:", len(latest), "/", EXPECTED_GENERATIONS)
if len(latest) != EXPECTED_GENERATIONS:
    raise RuntimeError(
        "Scoring requires all 6,224 successful keys. Return to the generation cell to resume."
    )

CONDITIONS = ["UNSTEERED", "ASSISTANT_AXIS_CAP_SOURCE_SETTING"]

# ---------- Official IFEval ----------
assert ensure_google_research(ref=freeze_meta["google_research_commit"]) == freeze_meta["google_research_commit"]

if str(GOOGLE_RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(GOOGLE_RESEARCH_DIR))

from instruction_following_eval import evaluation_lib as ifeval_lib

ifeval_path = GOOGLE_RESEARCH_DIR / "instruction_following_eval" / "data" / "input_data.jsonl"
if sha256_file(ifeval_path) != freeze_meta["ifeval_input_sha256"]:
    raise RuntimeError("IFEval input hash changed.")
ifeval_inputs = ifeval_lib.read_prompt_list(str(ifeval_path))
if len(ifeval_inputs) != N_IFEVAL:
    raise RuntimeError(f"Expected {N_IFEVAL} IFEval inputs, got {len(ifeval_inputs)}")

def score_ifeval(condition):
    prompt_to_response = {}
    for i, inp in enumerate(ifeval_inputs):
        row = latest[(f"ifeval-{i:03d}", condition)]
        prompt_to_response[inp.prompt] = row["completion"]

    strict_outputs = [
        ifeval_lib.test_instruction_following_strict(inp, prompt_to_response)
        for inp in ifeval_inputs
    ]
    loose_outputs = [
        ifeval_lib.test_instruction_following_loose(inp, prompt_to_response)
        for inp in ifeval_inputs
    ]

    out_dir = FULL_ROOT / f"ifeval_{condition.lower()}"
    out_dir.mkdir(parents=True, exist_ok=True)
    ifeval_lib.write_outputs(str(out_dir / "eval_results_strict.jsonl"), strict_outputs)
    ifeval_lib.write_outputs(str(out_dir / "eval_results_loose.jsonl"), loose_outputs)

    def agg(outputs):
        prompt_flags = [bool(o.follow_all_instructions) for o in outputs]
        instruction_flags = [
            bool(flag)
            for o in outputs
            for flag in o.follow_instruction_list
        ]
        return {
            "prompt_accuracy": float(np.mean(prompt_flags)),
            "instruction_accuracy": float(np.mean(instruction_flags)),
            "prompt_n": len(prompt_flags),
            "instruction_n": len(instruction_flags),
        }

    return {"strict": agg(strict_outputs), "loose": agg(loose_outputs)}

ifeval_scores = {c: score_ifeval(c) for c in CONDITIONS}

# ---------- MMLU-Pro / GSM8K ----------
def binary_accuracy(benchmark, condition):
    rows = [
        latest[(x["item_id"], condition)]
        for x in frozen_items if x["benchmark"] == benchmark
    ]
    return {
        "accuracy": float(np.mean([bool(r.get("correct")) for r in rows])),
        "n": len(rows),
        "parseable_n": int(sum(bool(r.get("parseable")) for r in rows)),
        "parseable_fraction": float(np.mean([bool(r.get("parseable")) for r in rows])),
    }

mmlu_scores = {c: binary_accuracy("MMLU-Pro", c) for c in CONDITIONS}
gsm_scores = {c: binary_accuracy("GSM8K", c) for c in CONDITIONS}

# ---------- Official EQ-Bench v2 source functions ----------
assert ensure_exact_repo(
    EQBENCH_REPO, EQBENCH_DIR,
    ref=freeze_meta["eqbench_commit"],
) == freeze_meta["eqbench_commit"]

eq_scoring_path = EQBENCH_DIR / "lib" / "scoring.py"
if sha256_file(eq_scoring_path) != freeze_meta["eqbench_scoring_sha256"]:
    raise RuntimeError("EQ-Bench scoring.py hash differs from frozen provenance.")

eq_parse_answers = load_function_from_source(
    eq_scoring_path, "parse_answers", globals_dict={"re": re}
)
eq_calculate_score_fullscale = load_function_from_source(
    eq_scoring_path, "calculate_score_fullscale", globals_dict={"math": math}
)

def score_eq(condition):
    per_item = []
    parseable = 0
    for item in [x for x in frozen_items if x["benchmark"] == "EQ-Bench"]:
        row = latest[(item["item_id"], condition)]
        answers, _ = eq_parse_answers(row["completion"], False)
        ref = item["native_metadata"]["reference_answer_fullscale"]
        s = eq_calculate_score_fullscale(ref, answers)
        if s is not None:
            parseable += 1
            per_item.append(float(s))

    # Official v2 aggregation: 100 * mean(item_score / 10) over parseable answers.
    score_100 = 100.0 * (float(np.mean(per_item)) / 10.0) if per_item else None
    return {
        "score": round(score_100, 2) if score_100 is not None else None,
        "n": N_EQ,
        "parseable_n": parseable,
        "parseable_fraction": parseable / N_EQ,
    }

eq_scores = {c: score_eq(c) for c in CONDITIONS}

primary = {
    "IFEval": {
        c: ifeval_scores[c]["strict"]["prompt_accuracy"] for c in CONDITIONS
    },
    "MMLU-Pro": {
        c: mmlu_scores[c]["accuracy"] for c in CONDITIONS
    },
    "GSM8K": {
        c: gsm_scores[c]["accuracy"] for c in CONDITIONS
    },
    "EQ-Bench": {
        c: eq_scores[c]["score"] for c in CONDITIONS
    },
}

summary_rows = []
for benchmark in ["IFEval", "MMLU-Pro", "GSM8K", "EQ-Bench"]:
    u = primary[benchmark]["UNSTEERED"]
    c = primary[benchmark]["ASSISTANT_AXIS_CAP_SOURCE_SETTING"]
    if u is None or c is None:
        raise RuntimeError(f"Missing primary metric for {benchmark}")
    u = float(u)
    c = float(c)
    rel = ((u - c) / u * 100.0) if u != 0 else None
    summary_rows.append({
        "benchmark": benchmark,
        "primary_metric": (
            "strict_prompt_accuracy"
            if benchmark == "IFEval"
            else "accuracy"
            if benchmark in {"MMLU-Pro", "GSM8K"}
            else "eqbench_v2_score_0_100"
        ),
        "n": {
            "IFEval": N_IFEVAL,
            "MMLU-Pro": N_MMLU,
            "GSM8K": N_GSM,
            "EQ-Bench": N_EQ,
        }[benchmark],
        "unsteered": u,
        "capped": c,
        "absolute_delta_capped_minus_unsteered": c - u,
        "relative_performance_reduction_pct": rel,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(SUMMARY_PATH, index=False)

reduction_values = [
    r["relative_performance_reduction_pct"]
    for r in summary_rows
    if r["relative_performance_reduction_pct"] is not None
]
aggregate_reduction = float(sum(reduction_values))

report = {
    "created_at_utc": utcnow(),
    "status": "COMPLETE_LU_SIZED_PUBLIC_CAPABILITY_BATTERY",
    "fidelity_label": freeze_meta["fidelity_label"],
    "exact_lu_membership_claim_allowed": False,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "precision": "bfloat16",
    "quantized": False,
    "thinking_enabled": False,
    "source_repo_commit": SOURCE_COMMIT,
    "capping_dataset_revision": CAP_DATASET_REVISION,
    "capping_config_sha256": CAP_FILE_SHA256,
    "capping_experiment": CAP_EXPERIMENT,
    "manifest_sha256": freeze_meta["manifest_sha256"],
    "successful_generations": len(latest),
    "IFEval": ifeval_scores,
    "MMLU-Pro": mmlu_scores,
    "GSM8K": gsm_scores,
    "EQ-Bench": eq_scores,
    "summary_rows": summary_rows,
    "sum_of_relative_performance_reductions_pct": aggregate_reduction,
    "claim_boundary": (
        "Matches Lu-reported capability benchmark families and sample sizes. "
        "MMLU-Pro/GSM8K exact Lu row membership and exact Assistant Axis capability "
        "prompt/decoding are not established; do not call this an exact Lu replication."
    ),
}
json_dump_atomic(report, REPORT_PATH)

print("\nIFEval:", json.dumps(ifeval_scores, indent=2))
print("MMLU-Pro:", json.dumps(mmlu_scores, indent=2))
print("GSM8K:", json.dumps(gsm_scores, indent=2))
print("EQ-Bench:", json.dumps(eq_scores, indent=2))
print("\nSum of relative performance reductions (%):", aggregate_reduction)
print("\nSaved:", SUMMARY_PATH)
print("Saved:", REPORT_PATH)


## Done

The run is complete only after the final cell shows:

`Scorable successful keys: 6224 / 6224`

and writes:

- `MyDrive/[anonymized-repository-name]-t21/full_capability/t21_full_capability_summary.csv`
- `MyDrive/[anonymized-repository-name]-t21/full_capability/t21_full_capability_report.json`

If Colab disconnects, rerun from the top on another high-memory GPU. The manifest remains frozen and successful generation keys are skipped automatically.